In [1]:
import re
import math
import numpy as np
import pandas as pd
from pathlib import Path


PROCESSED_ROOT = Path("/mnt/nas/trading_project/data/processed/Options")
SPX_1DAY_PATH = Path(
    "/mnt/nas/trading_project/data/unprocessed/First Rate Data/index/1day/SPX_full_1day.txt"
)

def load_spx_index_options(years):
    dfs = []
    for y in years:
        path = PROCESSED_ROOT / str(y) / f"spx_index_options_{y}.parquet"
        df = pd.read_parquet(path)
        df["year"] = y
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def load_spx_index(start_date="2010-01-01"):
    date_re = re.compile(r"^(\d{4}[-/]\d{2}[-/]\d{2}|\d{2}/\d{2}/\d{4}|\d{8})")

    rows = []
    with SPX_1DAY_PATH.open("r", errors="replace") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            if not date_re.match(s):
                continue

            parts = re.split(r"[,\t;|]+|\s{2,}", s)
            if len(parts) >= 6:
                rows.append(parts[:6])

    df = pd.DataFrame(rows, columns=["date", "open", "high", "low", "close", "volume"])
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.tz_localize(None)

    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df[df["date"].notna()].sort_values("date").reset_index(drop=True)
    df = df[df["date"] >= pd.to_datetime(start_date)].reset_index(drop=True)
    return df


def _pick_mid_price(
    df: pd.DataFrame,
    mid_col="mid",
    bid_col="bid",
    ask_col="ask",
    last_col="last",
    price_col="price",
) -> pd.Series:

    cols = df.columns

    if mid_col in cols:
        return pd.to_numeric(df[mid_col], errors="coerce")

    if bid_col in cols and ask_col in cols:
        bid = pd.to_numeric(df[bid_col], errors="coerce")
        ask = pd.to_numeric(df[ask_col], errors="coerce")
        return (bid + ask) / 2.0

    if last_col in cols:
        return pd.to_numeric(df[last_col], errors="coerce")

    if price_col in cols:
        return pd.to_numeric(df[price_col], errors="coerce")

    raise ValueError("Non trovo colonne prezzo valide.")


def estimate_r_and_forward_from_parity(
    df_raw: pd.DataFrame,
    option_type_col="option_type",
    strike_col="strike",
    trade_date_col="trade_date",
    expiration_col="expiration",
    dte_col="dte",
    min_strikes=5,
    year_basis=365.0,
    mid_kwargs=None,
) -> pd.DataFrame:

    if mid_kwargs is None:
        mid_kwargs = {}

    df_work = df_raw.copy()
    df_work["mid"] = _pick_mid_price(df_work, **mid_kwargs)

    use_cols = [
        trade_date_col,
        expiration_col,
        strike_col,
        option_type_col,
        dte_col,
        "mid",
    ]
    df0 = df_work[use_cols].copy()

    ot = df0[option_type_col].astype(str).str.lower().str.strip()

    df0["opt"] = np.where(
        ot.str.startswith("c"), "call",
        np.where(ot.str.startswith("p"), "put", None)
    )

    df0["mid"] = pd.to_numeric(df0["mid"], errors="coerce")
    df0[strike_col] = pd.to_numeric(df0[strike_col], errors="coerce")
    df0[dte_col] = pd.to_numeric(df0[dte_col], errors="coerce")

    df0 = df0.dropna(
        subset=["opt", "mid", strike_col, trade_date_col, expiration_col, dte_col]
    )

    piv = (
        df0.pivot_table(
            index=[trade_date_col, expiration_col, strike_col, dte_col],
            columns="opt",
            values="mid",
            aggfunc="median",
        )
        .reset_index()
    )

    piv["call"] = pd.to_numeric(piv.get("call"), errors="coerce")
    piv["put"] = pd.to_numeric(piv.get("put"), errors="coerce")

    piv = piv.dropna(subset=["call", "put"]).copy()
    piv["cp"] = piv["call"] - piv["put"]

    piv["T"] = pd.to_numeric(piv[dte_col], errors="coerce") / year_basis
    piv = piv[piv["T"] > 0]

    out_rows = []

    for (td, exp), g in piv.groupby([trade_date_col, expiration_col], sort=False):

        n = int(pd.Series(g[strike_col]).nunique())
        if n < min_strikes:
            continue

        T = float(g["T"].median())

        K = pd.to_numeric(g[strike_col], errors="coerce").to_numpy(float)
        y = pd.to_numeric(g["cp"], errors="coerce").to_numpy(float)

        mask = np.isfinite(K) & np.isfinite(y)
        if mask.sum() < min_strikes:
            continue

        K = K[mask]
        y = y[mask]

        X = np.column_stack([np.ones_like(K), K])
        a, b = np.linalg.lstsq(X, y, rcond=None)[0]

        DF = -b
        if not np.isfinite(DF) or DF <= 0:
            continue

        F = a / DF
        if not np.isfinite(F) or F <= 0:
            continue

        r = -np.log(DF) / T

        out_rows.append(
            {
                trade_date_col: td,
                expiration_col: exp,
                "T": T,
                "df": DF,
                "r": r,
                "forward": F,
                "n_strikes": int(mask.sum()),
            }
        )

    return (
        pd.DataFrame(out_rows)
        .sort_values([trade_date_col, expiration_col])
        .reset_index(drop=True)
    )


def attach_forward_curve(
    df_filt: pd.DataFrame,
    fwd_df: pd.DataFrame,
    trade_date_col="trade_date",
    expiration_col="expiration",
) -> pd.DataFrame:

    return df_filt.merge(
        fwd_df[[trade_date_col, expiration_col, "T", "df", "r", "forward", "n_strikes"]],
        on=[trade_date_col, expiration_col],
        how="left",
        validate="many_to_one",
    )


def add_iv_mid(df: pd.DataFrame, bid_iv_col="bid_iv", ask_iv_col="ask_iv", out_col="iv_mid"):
    out = df.copy()
    out[out_col] = 0.5 * (
        pd.to_numeric(out[bid_iv_col], errors="coerce")
        + pd.to_numeric(out[ask_iv_col], errors="coerce")
    )
    return out


def add_forward_moneyness(df: pd.DataFrame, strike_col="strike", forward_col="forward", out_col="log_moneyness_fwd"):
    out = df.copy()
    K = pd.to_numeric(out[strike_col], errors="coerce")
    F = pd.to_numeric(out[forward_col], errors="coerce")
    out[out_col] = np.log(K / F)
    return out


def build_df_fwd(
    years_opt,
    spx_start="2010-01-01",
    max_dte=220,
    n_atm_strikes=11,
):
    df_opt = load_spx_index_options(years_opt)
    df_spx = load_spx_index(start_date=spx_start)

    df_raw = df_opt.merge(
        df_spx[["date", "close"]].rename(columns={"date": "trade_date", "close": "spx_close"}),
        on="trade_date",
        how="left",
        validate="many_to_one",
    )
    df_raw = df_raw.dropna(subset=["spx_close"])

    df_filt = df_raw[df_raw["dte"] <= max_dte].copy()

    strike_level = df_filt[["trade_date", "expiration", "strike", "spx_close"]].drop_duplicates()
    strike_level["atm_dist"] = (
        pd.to_numeric(strike_level["strike"], errors="coerce")
        - pd.to_numeric(strike_level["spx_close"], errors="coerce")
    ).abs()

    atm_strikes = (
        strike_level.sort_values(["trade_date", "expiration", "atm_dist"])
        .groupby(["trade_date", "expiration"], sort=False)
        .head(n_atm_strikes)[["trade_date", "expiration", "strike"]]
    )

    df_filt = df_filt.merge(
        atm_strikes,
        on=["trade_date", "expiration", "strike"],
        how="inner",
        validate="many_to_one",
    )

    fwd_curve = estimate_r_and_forward_from_parity(
        df_raw=df_filt,
        option_type_col="option_type",
        strike_col="strike",
        trade_date_col="trade_date",
        expiration_col="expiration",
        dte_col="dte",
        min_strikes=5,
        year_basis=365.0,
        mid_kwargs={
            "mid_col": "mid",
            "bid_col": "bid",
            "ask_col": "ask",
            "last_col": "last",
            "price_col": "price",
        },
    )

    df_fwd = attach_forward_curve(df_filt, fwd_curve)

    df_fwd = add_iv_mid(df_fwd, bid_iv_col="bid_iv", ask_iv_col="ask_iv", out_col="iv_mid")
    df_fwd = add_forward_moneyness(df_fwd, strike_col="strike", forward_col="forward", out_col="log_moneyness_fwd")

    return df_fwd


import gc

years_opt = list(range(2010, 2026))

dfs = []

for y in years_opt:
    print(f"\n=== Processing year {y} ===")

    try:
        df_y = build_df_fwd(
            years_opt=[y],
            spx_start=f"{y}-01-01",
            max_dte=220,
            n_atm_strikes=11,
        )

        print(f"Year {y} done | shape = {df_y.shape}")

        dfs.append(df_y)

        # pulizia memoria aggressiva
        del df_y
        gc.collect()

    except Exception as e:
        print(f"⚠️ Year {y} FAILED: {e}")
        continue

# concat finale
df_fwd = (
    pd.concat(dfs, ignore_index=True)
      .sort_values(["trade_date", "expiration", "strike"])
      .reset_index(drop=True)
)

print("\n=== FINAL df_fwd ===")
print(df_fwd.shape)
print(df_fwd.head())



=== Processing year 2010 ===
⚠️ Year 2010 FAILED: [Errno 2] No such file or directory: '/mnt/nas/trading_project/data/processed/Options/2010/spx_index_options_2010.parquet'

=== Processing year 2011 ===
⚠️ Year 2011 FAILED: [Errno 2] No such file or directory: '/mnt/nas/trading_project/data/processed/Options/2011/spx_index_options_2011.parquet'

=== Processing year 2012 ===
⚠️ Year 2012 FAILED: [Errno 2] No such file or directory: '/mnt/nas/trading_project/data/processed/Options/2012/spx_index_options_2012.parquet'

=== Processing year 2013 ===
⚠️ Year 2013 FAILED: [Errno 2] No such file or directory: '/mnt/nas/trading_project/data/processed/Options/2013/spx_index_options_2013.parquet'

=== Processing year 2014 ===
⚠️ Year 2014 FAILED: [Errno 2] No such file or directory: '/mnt/nas/trading_project/data/processed/Options/2014/spx_index_options_2014.parquet'

=== Processing year 2015 ===
⚠️ Year 2015 FAILED: [Errno 2] No such file or directory: '/mnt/nas/trading_project/data/processed/O

ValueError: No objects to concatenate

In [ ]:
#Greeks calculus

from scipy.stats import norm

def black76_d1_d2(F, K, T, sigma):
    eps = 1e-12
    sigma = np.maximum(sigma, eps)
    T = np.maximum(T, eps)

    vol_sqrtT = sigma * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / vol_sqrtT
    d2 = d1 - vol_sqrtT
    return d1, d2

def black76_greeks(F, K, T, r, sigma, option_type):
    df = np.exp(-r * T)
    eps = 1e-12
    F = np.maximum(F, eps)
    K = np.maximum(K, eps)
    T = np.maximum(T, eps)
    sigma = np.maximum(sigma, eps)

    vol_sqrtT = sigma * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / vol_sqrtT
    d2 = d1 - vol_sqrtT

    is_call = option_type.astype(str).str.upper().str.startswith("C").values

    pdf_d1 = norm.pdf(d1)      # ✅ FIX
    cdf_d1 = norm.cdf(d1)
    cdf_d2 = norm.cdf(d2)

    delta = np.where(is_call, df * cdf_d1, -df * norm.cdf(-d1))

    gamma = df * pdf_d1 / (F * sigma * np.sqrt(T))

    vega = df * F * pdf_d1 * np.sqrt(T) * 0.01  # per 1% vol

    theta = (
        -df * F * pdf_d1 * sigma / (2.0 * np.sqrt(T))
        + r * df * np.where(
            is_call,
            F * cdf_d1 - K * cdf_d2,
            -F * norm.cdf(-d1) + K * norm.cdf(-d2),
        )
    ) / 365.0

    return delta, gamma, vega, theta

def build_black76_greeks_df(df_fwd):
    df = df_fwd.copy()

    needed = [
        "trade_date", "expiration",
        "strike", "forward", "T", "r",
        "iv_mid", "option_type",
    ]

    df = df.dropna(subset=needed)

    delta, gamma, vega, theta = black76_greeks(
        F=df["forward"].values,
        K=df["strike"].values,
        T=df["T"].values,
        r=df["r"].values,
        sigma=df["iv_mid"].values,
        option_type=df["option_type"],
    )

    df_greeks = df[[
        "trade_date",
        "expiration",
        "option_type",
        "strike",
        "T",
        "forward",
        "iv_mid",
    ]].copy()

    df_greeks["delta"] = delta
    df_greeks["gamma"] = gamma
    df_greeks["vega"]  = vega
    df_greeks["theta"] = theta

    return df_greeks

df_greeks = build_black76_greeks_df(df_fwd)


In [ ]:
#load VIX continuous absolute

import os
import pandas as pd

def load_vx_future_continuous_absolute(
    path,
    start_date="2010-01-01",
    sep=",",
    date_col=0,
    price_col=4,
):
    df = pd.read_csv(path, sep=sep, header=None)

    df = df[[date_col, price_col]].copy()
    df.columns = ["date", "vx"]

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["vx"] = pd.to_numeric(df["vx"], errors="coerce")

    df = df.dropna().sort_values("date").reset_index(drop=True)
    df = df[df["date"] >= pd.to_datetime(start_date)]

    return df


VX_PATH = "/mnt/c/Users/marco/Downloads/futures_full_1day_contin_UNadj/VX_full_1day_continuous_UNadjusted.txt"

df_vx = load_vx_future_continuous_absolute(VX_PATH)

print(df_vx.head())
print(df_vx.tail())
print(len(df_vx))






In [ ]:
#load VIX continuous absolute

import os
import pandas as pd

def load_vx_future_continuous_absolute(
    path,
    start_date="2010-01-01",
    sep=",",
    date_col=0,
    price_col=4,
):
    df = pd.read_csv(path, sep=sep, header=None)

    df = df[[date_col, price_col]].copy()
    df.columns = ["date", "vx"]

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["vx"] = pd.to_numeric(df["vx"], errors="coerce")

    df = df.dropna().sort_values("date").reset_index(drop=True)
    df = df[df["date"] >= pd.to_datetime(start_date)]

    return df


VX_PATH = "/mnt/c/Users/marco/Downloads/futures_full_1day_contin_adj_absolute/VX_full_1day_continuous_absolute_adjusted.txt"

df_vx = load_vx_future_continuous_absolute(VX_PATH)

print(df_vx.head())
print(df_vx.tail())
print(len(df_vx))






In [ ]:
#Term structure building

PARQUET_PATH = Path("/mnt/c/Users/Marco/Downloads/df_fwd_full.parquet")

df = pd.read_parquet(PARQUET_PATH)

cols = [
    "trade_date",
    "expiration",
    "dte",
    "iv_mid",
    "vega",
    "log_moneyness_fwd",
    "ticker"
]
df = df[cols].copy()
df = df[df["ticker"] == "SPX"].copy()

df["trade_date"] = pd.to_datetime(df["trade_date"])
df["expiration"] = pd.to_datetime(df["expiration"])

# basic cleaning
df = df.dropna(subset=["expiration", "dte", "iv_mid", "vega", "log_moneyness_fwd"])
df = df[(df["iv_mid"] > 0) & (df["vega"] > 0) & (df["dte"] > 0)]

# variance
df["variance"] = df["iv_mid"] ** 2

TARGET_DTES = [14, 30, 60, 90, 180]
# Nota: nessuna soglia ATM qui

records = []

# Per ogni giorno:
for trade_date, g_day in df.groupby("trade_date"):
    # 1) Costruisci, per ogni expiration, un punto ATM "reale"
    #    ATM = moneyness più vicino a 0 in quella expiration (forward-based)
    exp_points = []
    for exp, g_exp in g_day.groupby("expiration"):
        # prendi le righe con moneyness assoluta minima (ATM reale)
        m_abs = np.abs(g_exp["log_moneyness_fwd"])
        m_min = m_abs.min()
        atm_slice = g_exp[m_abs == m_min]

        # vega-weighted variance per quel slice ATM
        var_atm = np.sum(atm_slice["variance"] * atm_slice["vega"]) / np.sum(atm_slice["vega"])
        dte_rep = float(np.median(g_exp["dte"]))  # DTE rappresentativo della expiration

        exp_points.append((exp, dte_rep, var_atm))

    if len(exp_points) == 0:
        continue

    exp_df = pd.DataFrame(exp_points, columns=["expiration", "dte_rep", "variance_atm"])

    # 2) Per ciascun target DTE scegli la expiration reale più vicina
    for target in TARGET_DTES:
        idx = (exp_df["dte_rep"] - target).abs().idxmin()
        row = exp_df.loc[idx]

        records.append({
            "trade_date": trade_date,
            "target_dte": target,
            "actual_expiration": row["expiration"],
            "actual_dte": row["dte_rep"],
            "variance": row["variance_atm"]
        })

ts_long = pd.DataFrame(records)

ts_panel = ts_long.pivot(index="trade_date", columns="target_dte", values="variance").sort_index()
ts_panel.columns = [f"var_{c}d" for c in ts_panel.columns]

ts_panel.head()




In [ ]:
#PCA model for fitting the term structure

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# log-variance
X = np.log(ts_panel).dropna()

# centra (IMPORTANTISSIMO per PCA)
X_mean = X.mean()
Xc = X - X_mean

pca = PCA(n_components=3)
F = pca.fit_transform(Xc)      # fattori nel tempo (T x 3)
B = pca.components_            # loadings (3 x 5)

plt.figure(figsize=(6,4))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.axhline(0.95, color="r", linestyle="--", label="95%")
plt.title("PCA – Explained Variance (cumulata)")
plt.ylabel("Quota di varianza spiegata")
plt.xlabel("Numero di fattori")
plt.legend()
plt.tight_layout()
plt.show()

X_hat = F @ B + X_mean.values
X_hat = pd.DataFrame(X_hat, index=X.index, columns=X.columns)

fig, axes = plt.subplots(2, 3, figsize=(14,8))
axes = axes.flatten()

for i, col in enumerate(X.columns):
    ax = axes[i]
    ax.scatter(X[col], X_hat[col], alpha=0.3)
    ax.plot([X[col].min(), X[col].max()],
            [X[col].min(), X[col].max()],
            color="red", linestyle="--")
    ax.set_title(col)
    ax.set_xlabel("True log-variance")
    ax.set_ylabel("Reconstructed")

plt.suptitle("PCA (3 fattori) – Accuratezza di ricostruzione in-sample")
plt.tight_layout()
plt.show()

recon_error = (X - X_hat)

rmse_by_maturity = np.sqrt((recon_error**2).mean())
print("RMSE per maturity (log-variance):")
print(rmse_by_maturity)

plt.figure(figsize=(6,4))
for i in range(3):
    plt.plot(X.columns, B[i], marker="o", label=f"PC{i+1}")
plt.title("PCA Loadings (forma dei fattori)")
plt.ylabel("Loading")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

for i in range(3):
    axes[i].plot(X.index, F[:, i])
    axes[i].set_title(f"PCA Factor {i+1}")

plt.tight_layout()
plt.show()

from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(3, 1, figsize=(10, 8))

for i in range(3):
    plot_acf(F[:, i], lags=30, ax=axes[i])
    axes[i].set_title(f"ACF PCA Factor {i+1}")

plt.tight_layout()
plt.show()

from statsmodels.tsa.stattools import adfuller

for i in range(3):
    stat, pval, *_ = adfuller(F[:, i])
    print(f"PC{i+1} | ADF p-value: {pval:.4f}")

import numpy as np
import statsmodels.api as sm

def half_life(series):
    y = series[1:]
    x = series[:-1]
    x = sm.add_constant(x)
    res = sm.OLS(y, x).fit()
    phi = res.params[1]
    return -np.log(2) / np.log(phi)

for i in range(3):
    hl = half_life(F[:, i])
    print(f"PC{i+1} half-life: {hl:.1f} giorni")

In [ ]:
H = 5
WINDOW = 504
N_FACTORS = 3
USE_FACTORS = [0, 1]

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import statsmodels.api as sm

X = np.log(ts_panel).dropna()
dates = X.index
cols = X.columns
T = len(X)

def rmse(a, b):
    a = np.asarray(a); b = np.asarray(b)
    return np.sqrt(np.mean((a-b)**2))

h_eval = H - 1  # <-- valutiamo t+H

F_true_list, F_hat_list = [], []
X_true_list, X_hat_list = [], []

for t in range(WINDOW, T - H):
    train = X.iloc[t-WINDOW:t]     # fino a t-1
    test  = X.iloc[t:t+H]          # t..t+H-1
    
    mu = train.mean()
    pca = PCA(n_components=N_FACTORS).fit(train - mu)
    
    F_train = pca.transform(train - mu)
    F_test  = pca.transform(test  - mu)

    # AR(1) forecast iterativo
    F_fore = np.zeros((H, N_FACTORS))
    for k in USE_FACTORS:
        y = F_train[1:, k]
        x = F_train[:-1, k]
        res = sm.OLS(y, sm.add_constant(x)).fit()
        a, b = res.params

        x0 = F_train[-1, k]
        for h in range(H):
            x1 = a + b*x0
            F_fore[h, k] = x1
            x0 = x1

    X_hat = (F_fore @ pca.components_) + mu.values  # (H x n_maturities)

    # <-- salva l'orizzonte H (non h=0)
    F_true_list.append(F_test[h_eval])
    F_hat_list.append(F_fore[h_eval])
    X_true_list.append(test.iloc[h_eval].values)
    X_hat_list.append(X_hat[h_eval])

# allinea le date ai target t+H
bt_dates = dates[WINDOW + (H-1) :][ : len(F_true_list) ]

F_true = pd.DataFrame(F_true_list, index=bt_dates, columns=[f"PC{i+1}" for i in range(N_FACTORS)])
F_hat  = pd.DataFrame(F_hat_list,  index=bt_dates, columns=[f"PC{i+1}_hat" for i in range(N_FACTORS)])

X_true = pd.DataFrame(X_true_list, index=bt_dates, columns=cols)
X_hat  = pd.DataFrame(X_hat_list,  index=bt_dates, columns=[c+"_hat" for c in cols])

print("\n=== Accuracy fattori (H-step) ===")
for i in range(N_FACTORS):
    f = F_true[f"PC{i+1}"].values
    g = F_hat[f"PC{i+1}_hat"].values
    print(f"PC{i+1}: RMSE={rmse(f,g):.4f}  Corr={np.corrcoef(f,g)[0,1]:.3f}")

print("\n=== Shape accuracy (H-step) ===")
slope_true = X_true["var_14d"] - X_true["var_180d"]
slope_hat  = X_hat["var_14d_hat"] - X_hat["var_180d_hat"]
print("Slope Corr:", np.corrcoef(slope_true, slope_hat)[0,1])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# regime split su PC1 (livello)
pc1 = F_true["PC1"]
pc1_q = pc1.quantile([0.33, 0.66])

regime = pd.cut(
    pc1,
    bins=[-np.inf, pc1_q.iloc[0], pc1_q.iloc[1], np.inf],
    labels=["Low level", "Mid level", "High level"]
)

plt.figure(figsize=(7,6))
for r, c in zip(["Low level","Mid level","High level"], ["green","orange","red"]):
    idx = regime == r
    plt.scatter(
        slope_true[idx],
        slope_hat[idx],
        alpha=0.4,
        label=r,
        color=c
    )

mn = min(slope_true.min(), slope_hat.min())
mx = max(slope_true.max(), slope_hat.max())
plt.plot([mn, mx], [mn, mx], "k--")

plt.xlabel("Realized slope (log-var)")
plt.ylabel("Forecast slope (log-var)")
plt.title("Slope forecast vs realized — colored by PC1 regime")
plt.legend()
plt.tight_layout()
plt.show()

error = slope_hat - slope_true

plt.figure(figsize=(7,5))
plt.scatter(pc1, error, alpha=0.4)
plt.axhline(0, color="k", linestyle="--")
plt.xlabel("PC1 (level)")
plt.ylabel("Forecast error (slope)")
plt.title("Slope forecast error vs level (PC1)")
plt.tight_layout()
plt.show()

wrong_sign = np.sign(slope_hat) != np.sign(slope_true)

plt.figure(figsize=(7,5))
plt.scatter(
    pc1[~wrong_sign],
    slope_true[~wrong_sign],
    alpha=0.3,
    label="Correct sign",
    color="blue"
)
plt.scatter(
    pc1[wrong_sign],
    slope_true[wrong_sign],
    alpha=0.6,
    label="Wrong sign",
    color="red"
)

plt.axhline(0, color="k", linestyle="--")
plt.xlabel("PC1 (level)")
plt.ylabel("Realized slope")
plt.title("Wrong-sign forecasts vs level")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,5))
plt.scatter(np.abs(slope_hat), np.abs(error), alpha=0.4)
plt.xlabel("|Forecast slope|")
plt.ylabel("|Forecast error|")
plt.title("Error vs signal strength")
plt.tight_layout()
plt.show()

rw_hat = slope_true.shift(1).loc[slope_hat.index]

plt.figure(figsize=(7,6))
plt.scatter(
    rw_hat - slope_true,
    slope_hat - slope_true,
    alpha=0.4
)
plt.axhline(0, color="k", linestyle="--")
plt.axvline(0, color="k", linestyle="--")
plt.xlabel("RW error")
plt.ylabel("AR(1) error")
plt.title("AR(1) vs RW errors (slope)")
plt.tight_layout()
plt.show()



In [ ]:

# ----------------------------
# 1) Segnale base (slope / PC2)
# ----------------------------
signal_raw = slope_hat.copy()

# normalizzazione (stabile)
signal_z = signal_raw / signal_raw.rolling(60).std()

# soglia per evitare rumore
signal_ok = signal_z.abs() > 0.5

# ----------------------------
# 2) Filtro di regime PC1
# ----------------------------
pc1 = F_true["PC1"]

pc1_z = (pc1 - pc1.rolling(252).mean()) / pc1.rolling(252).std()
regime_ok = pc1_z < 1.0   # stress-off filter

# ----------------------------
# 3) Segnali finali
# ----------------------------
signal_nofilter = signal_z * signal_ok
signal_filtered = signal_z * signal_ok * regime_ok

# ----------------------------
# 4) Metriche
# ----------------------------
def hit_rate(pred, true):
    idx = pred != 0
    return (np.sign(pred[idx]) == np.sign(true[idx])).mean()

def masked_corr(pred, true):
    idx = pred != 0
    return np.corrcoef(pred[idx], true[idx])[0,1]

print("===== PRODUZIONE FORECAST =====")
print(f"Coverage NO filter: {(signal_nofilter!=0).mean():.2%}")
print(f"Coverage FILTERED : {(signal_filtered!=0).mean():.2%}")
print()
print(f"Hit-rate NO filter: {hit_rate(signal_nofilter, slope_true):.3f}")
print(f"Hit-rate FILTERED : {hit_rate(signal_filtered, slope_true):.3f}")
print()
print(f"Corr NO filter    : {masked_corr(signal_nofilter, slope_true):.3f}")
print(f"Corr FILTERED     : {masked_corr(signal_filtered, slope_true):.3f}")

# ----------------------------
# 5) Plot time series
# ----------------------------
plt.figure(figsize=(12,4))
plt.plot(slope_true, label="Realized slope", color="black", alpha=0.5)
plt.plot(signal_nofilter, label="Signal NO filter", alpha=0.6)
plt.plot(signal_filtered, label="Signal FILTERED (PC1)", alpha=0.9)
plt.legend()
plt.title("Slope forecast – produzione con vs senza filtro PC1")
plt.tight_layout()
plt.show()

# ----------------------------
# 6) Scatter confronto
# ----------------------------
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.scatter(signal_nofilter, slope_true, alpha=0.3)
plt.axline((0,0),(1,1), linestyle="--")
plt.title("NO filter")
plt.xlabel("Forecast signal")
plt.ylabel("Realized slope")

plt.subplot(1,2,2)
idx = signal_filtered != 0
plt.scatter(signal_filtered[idx], slope_true[idx], alpha=0.3)
plt.axline((0,0),(1,1), linestyle="--")
plt.title("WITH PC1 filter")
plt.xlabel("Forecast signal")
plt.ylabel("Realized slope")

plt.tight_layout()
plt.show()


In [ ]:
# ======================================================
# VIX FUTURE STRATEGY
# PC2 (slope) signal + PC1 regime filter
# Rebalance every 5 days
# ======================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SIG_WIN = 60
PC1_WIN = 252
SIG_Z_TH = 0.5
PC1_Z_TH = 1.0
REBAL_STEP = 5

# ------------------------------------------------------
# 1) PREPARAZIONE VIX (ROBUSTA)
# ------------------------------------------------------
if "date" in df_vx.columns:
    df_vx = df_vx.set_index("date")

df_vx.index = pd.to_datetime(df_vx.index)

# ------------------------------------------------------
# 2) ALLINEAMENTO DATI
# ------------------------------------------------------
idx = (
    slope_hat.index
    .intersection(F_true.index)
    .intersection(df_vx.index)
)

vx = df_vx.loc[idx, "vx"].copy()
signal = slope_hat.loc[idx].copy()
pc1 = F_true.loc[idx, "PC1"].copy()

# ------------------------------------------------------
# 3) COSTRUZIONE SEGNALE
# ------------------------------------------------------
signal_z = signal / signal.rolling(SIG_WIN).std()
signal_ok = signal_z.abs() > SIG_Z_TH

pc1_z = (pc1 - pc1.rolling(PC1_WIN).mean()) / pc1.rolling(PC1_WIN).std()
regime_ok = pc1_z < PC1_Z_TH

final_signal = signal_z.where(signal_ok & regime_ok, 0.0)

# ------------------------------------------------------
# 4) POSIZIONE SU VIX
# ------------------------------------------------------
position = np.sign(final_signal)

position_5d = (
    position
    .iloc[::REBAL_STEP]
    .reindex(position.index, method="ffill")
    .shift(1)
    .fillna(0.0)
)

# ------------------------------------------------------
# 5) P&L
# ------------------------------------------------------
vx_ret = vx.diff()
pnl = (position_5d * vx_ret).fillna(0.0)
cum_pnl = pnl.cumsum()

# ------------------------------------------------------
# 6) METRICHE
# ------------------------------------------------------
def sharpe(x):
    if x.std() == 0:
        return np.nan
    return np.sqrt(252) * x.mean() / x.std()

def max_dd(cum):
    return (cum - cum.cummax()).min()

print("===== METRICHE STRATEGIA =====")
print("Sharpe:", round(sharpe(pnl), 2))
print("Max Drawdown:", round(max_dd(cum_pnl), 2))
print("Coverage (% giorni attivi):", round((position_5d != 0).mean() * 100, 1))

# ------------------------------------------------------
# 7) PLOT
# ------------------------------------------------------
plt.figure(figsize=(12,4))
cum_pnl.plot()
plt.title("VIX Strategy P&L (PC2 signal + PC1 filter, 5d rebalance)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
pnl.hist(bins=50)
plt.title("Distribution of daily P&L")
plt.tight_layout()
plt.show()

# ------------------------------------------------------
# 8) BENCHMARK: SEMPRE SHORT VIX
# ------------------------------------------------------
bench_pnl = -vx_ret.fillna(0.0)
bench_cum = bench_pnl.cumsum()

plt.figure(figsize=(12,4))
bench_cum.plot(label="Always short VIX")
cum_pnl.plot(label="Signal-driven")
plt.legend()
plt.title("Benchmark comparison")
plt.tight_layout()
plt.show()
print(cum_pnl.index.min(), cum_pnl.index.max())

